# 第 08 章 手工细胞类型注释

## 学习目标

把多个标记基因的证据转化为分层的细胞类型标签。

## 为什么做与怎样做

由程序导出当前聚类表达证据和空白建议表；AI 解释逐簇建议，用户确认后应用，绝不套用历史映射。

前置章节：07。运行前请完成项目环境准备。

本章在项目副本运行。遇到等待材料/确认，请按根目录 **99_运行与AI协作指南.md** 查看当前结果、与用户讨论并续跑；不能跳过待办直接进入下游。


In [ ]:
# 功能说明：查找公共环境和当前项目，读取有效上游检查点；模板副本之间不共享分析状态。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
from __future__ import annotations
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("SC_COURSE_ROOT", Path.cwd())).resolve()
while not (ROOT / "config/course.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config/course.json").is_file():
    raise RuntimeError("请从课程目录或其 notebooks/exercises 目录运行。")
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / ".runtime/celltypist")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".runtime/matplotlib")
sys.path.insert(0, str(ROOT / "tools"))
from course_runtime import start_chapter, marker_sets, scaled_view
from course_projects import resolve_project, input_path, read_sample
from course_resources import qc_gene_sets, marker_resources, resolution_preview
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
INSTALL_ROOT = ROOT
ROOT = resolve_project()
ctx = start_chapter("08")
adata = ctx.load_input()
coarse_key = str(adata.uns["annotation_keys"]["coarse"])
fine_key = str(adata.uns["annotation_keys"]["fine"])
sets = marker_sets(adata)
markers_1, markers_2 = sets["fine"], sets["broad"]
markers = markers_1
leiden_res = fine_key




## 08.1 类群注释

根据当前点图、UMAP 和表达比例为簇提出粗粒度标签。映射针对已经核对的这次聚类；新的数据和聚类结果需要重新核对。

下面读取已核对的映射，并使用 pandas.map 写入细胞注释。本次运行经用户确认的 annotation_proposal.csv 同时保存聚类指纹和标签，防止把一套聚类的编号用于另一套结果。

In [ ]:
# 变量/函数/参数解析：
# - ctx.annotation_map：首次调用导出当前表达证据、QC 与空白逐簇建议表。
# - AI 填写标签/理由并汇报；只有用户同意且表格与当前簇吻合，函数才返回映射。
# - manual_coarse：粗粒度标签；不会读取公共 config 的历史教程答案。
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
leiden_res = coarse_key
coarse_mapping = ctx.annotation_map(adata, leiden_res, "level1")
print(coarse_mapping)
adata.obs["manual_coarse"] = adata.obs[leiden_res].astype(str).map(coarse_mapping).astype("category")


In [ ]:
# 功能说明：查看细胞注释表。
# 运行目的：确认一级注释列 `manual_coarse` 是否成功添加。
# 详细代码解析：
# 1. `adata.obs`
#    - 打印概览。

adata.obs

In [ ]:
# 功能说明：重复绘制以便在不同分辨率下对比标签布局。
# 运行目的：辅助选择最终用于注释的分辨率。
sc.pl.umap(
    adata,
    color=[leiden_res, "manual_coarse", ],
    legend_loc="on data",
    save="_08_200.pdf",
)


In [ ]:
# 功能说明：统计一级注释的细胞数量。
# 运行目的：查看各主要细胞类型的分布情况。
# 详细代码解析：
# 1. `adata.obs["manual_coarse"].value_counts()`
#    - 统计一级注释列中各类别的频数。

# 查看每个类群细胞数量
adata.obs["manual_coarse"].value_counts()

## 08.2 更高分辨率下类群注释

In [ ]:
# 功能说明：切换到较高分辨率进行细致注释。
# 运行目的：为接下来的二级注释（小类）设置参数，使用分辨率 0.50 和详细标记基因集 `markers_1`。
# 详细代码解析：
# 1. `leiden_res = fine_key`
#    - 更新分辨率变量。
# 2. `markers = markers_1`
#    - 更新标记基因集变量。

# 定义聚类分辨率变量
leiden_res = fine_key
# 指定使用的markers版本
markers = markers_1

In [ ]:
# 功能说明：在较高分辨率簇上查看标记基因表达以细化注释。
# 运行目的：判断更细粒度的细胞类型分群。
sc.pl.dotplot(adata, markers, groupby=leiden_res, standard_scale="var", save="_08_204.pdf")

In [ ]:
# 功能说明：重复绘制以便在不同分辨率下对比标签布局。
# 运行目的：辅助选择最终用于注释的分辨率。
with rc_context({"figure.figsize": (10, 8)}):
    sc.pl.umap(
        adata,
        color=[leiden_res],
        legend_loc="on data",
    )

In [ ]:


# 使用 return_fig 参数返回图形对象
dot_plot = sc.pl.dotplot(
    adata, 
    markers, 
    groupby=leiden_res, 
    standard_scale="var",
    return_fig=True,
    show=False  # 不显示图形，只获取数据
)

# 检查可用的属性
print("可用的属性：", dir(dot_plot))

# 获取点图中的数据
if hasattr(dot_plot, 'dot_color_df'):
    print("颜色数据矩阵（标准化后的表达量）：")
    print(dot_plot.dot_color_df)
    
if hasattr(dot_plot, 'dot_size_df'):
    print("\n点大小数据矩阵（表达比例）：")
    print(dot_plot.dot_size_df)
    
# 最简单的方式：直接获取两个数据框
color_df = dot_plot.dot_color_df
size_df = dot_plot.dot_size_df
# 保存到CSV文件以便进一步分析
ctx.table("dotplot_color_fine", color_df)
ctx.table("dotplot_size_fine", size_df)

## 08.3 二级注释的证据组织

先按谱系做一级注释，再在谱系内部判断更细的类型。请同时考虑标记基因的表达比例、平均表达和在其他群体中的分布。单个标记的高表达不足以确认细胞类型。

建议整理包含聚类编号、一级标签、二级标签、支持标记、相反证据及待验证问题的表格。初始 B 细胞、抗体分泌细胞、T/NK 细胞和不同红系状态应分别查看相应的组合标记。对于不能清楚区分的亚型，先使用较宽泛或注明不确定性的标签。

下方展示当前参照映射及注释结果。证据表保存在 本章 results 中的 marker_evidence.csv 与 annotation_proposal.csv，具体参数与运行身份由 本次注释确认记录 关联。

In [ ]:
# 变量/函数/参数解析：
# - fine_level1/fine_level2：来自同一次已确认建议表的两层映射。
# - astype(str).map(...)：按当前簇编号映射；验证器确保没有漏簇或跨项目套用。
# - category：保留分类列类型，供 Scanpy 分组绘图和下游统计。
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
leiden_res = fine_key
fine_level1 = ctx.annotation_map(adata, leiden_res, "level1")
fine_level2 = ctx.annotation_map(adata, leiden_res, "level2")
print(fine_level2)
adata.obs["manual_level1"] = adata.obs[leiden_res].astype(str).map(fine_level1).astype("category")
adata.obs["manual_level2"] = adata.obs[leiden_res].astype(str).map(fine_level2).astype("category")


In [ ]:
# 功能说明：绘制多面板 UMAP 图，展示聚类和两级注释结果。
# 运行目的：直观对比聚类标签、一级注释和二级注释在空间上的分布。
# 详细代码解析：
# 1. `sc.pl.umap(...)`
#    - `color=[...]`: 同时展示聚类结果、二级注释、一级注释等多个视角。
#    - `save="..."`: 保存图像。

# 查看注释结果

sc.pl.umap(
        adata,
        color=[fine_key,  "manual_level2","manual_level1", "manual_coarse", ],
        legend_loc="on data",
        save="_08_209.pdf",
        )

In [ ]:
# 功能说明：单独绘制二级注释的 UMAP 图。
# 运行目的：清晰展示最终的细分细胞类型注释结果。
# 详细代码解析：
# 1. `with rc_context({"figure.figsize": (10, 8)}):`
#    - 临时设置图形大小为 10x8 英寸。
# 2. `sc.pl.umap(...)`
#    - `color=["manual_level2"]`: 仅展示二级注释。

# 查看注释结果
with rc_context({"figure.figsize": (10, 8)}):
    
    sc.pl.umap(
            adata,
            color=["manual_level2" ],
            legend_loc="on data",
            save="_08_210.pdf",
            )

In [ ]:
# 功能说明：绘制精美的多面板 UMAP 图，用于出版或展示。
# 运行目的：生成高质量的注释概览图，包含聚类、一级和二级注释，并进行详细的样式调整。
# 详细代码解析：
# 1. `with rc_context(...)`: 设置绘图上下文。
# 2. `sc.pl.umap(...)`:
#    - `color=[...]`: 展示三个层级的信息。
#    - `legend_loc`, `legend_fontsize`, `legend_fontoutline`: 优化图例显示。
#    - `size`, `add_outline`, `outline_width`, `outline_color`: 优化数据点样式，增加轮廓以提高区分度。
#    - `ncols`, `wspace`: 调整子图布局。
#    - `vmax`, `palette`: 调整颜色映射。
#    - `title`: 添加自定义标题。

with rc_context({"figure.figsize": (10, 8)}):
    sc.pl.umap(
        # 1. 数据对象
        adata,  # AnnData对象，包含单细胞数据和UMAP坐标
        
        # 2. 可视化内容：三个子图分别展示不同信息
        color=[
            leiden_res,              # 子图1: 分辨率0.50的Leiden聚类结果
            "manual_level1",      # 子图2: 一级细胞类型注释（基于分辨率0.50）
            "manual_level2",      # 子图3: 二级细胞类型注释（更细粒度的分类）
        ],
        # 3. 图例设置
        legend_loc="on data",               # 图例直接标注在数据点上，更直观
        legend_fontsize=10,                  # 减小字体大小以避免重叠（
     #   legend_fontweight='normal',         # 使用正常字体粗细，避免粗体占用过多空间
        legend_fontoutline=1.5,             # 添加1.5像素的字体描边，提高文字在复杂背景中的可读性
        # 4. 数据点设置
        size=25,                            # 数据点大小：平衡可视性和重叠问题（折中值）
        add_outline=True,                   # 为数据点添加轮廓，增强区分度
        outline_width=(0.4, 0.1),           # 轮廓宽度：外层0.4，内层0.1（更精细）
        outline_color=('black', 'white'),   # 轮廓颜色：外层黑色，内层白色，形成对比
        # 5. 子图布局
        ncols=3,                            # 3列布局（3个子图并排显示）
        wspace=0.6,                         # 子图之间的水平间距（0.6比0.5稍大，避免拥挤）
        
        # 6. 颜色映射设置
        vmax="p99",                         # 设置颜色映射最大值为第99百分位数，作用：去除极端值影响，使颜色分布更均匀
        palette="Set3",                     # 使用Set3调色板（适用于分类数据），Set3提供12种鲜明且易于区分的颜色
        # 7. 其他优化参数
       # frameon=False,                      # 去除子图边框，使图像更简洁
        show=False,                         # 不立即显示，便于后续调整或保存
        # 8. 新增参数优化
        alpha=0.8,                          # 设置透明度为0.8，在重叠区域显示更好
        edgecolor='none',                   # 去除边缘颜色，避免与轮廓冲突
        title=[                             # 为每个子图添加描述性标题
            f"A. Leiden Clusters ({fine_key})", 
            "B. Level 1 Cell Types", 
            "C. Level 2 Cell Types"
        ],
     #  save="_08_211.pdf",
    )
ctx.capture("manual_annotation_overview")


In [ ]:
# 功能说明：统计二级注释的细胞数量。
# 运行目的：查看各细分细胞类型的分布情况。
# 详细代码解析：
# 1. `adata.obs["manual_level2"].value_counts()`
#    - 统计二级注释列中各类别的频数。

# 查看每个类群细胞数量
adata.obs["manual_level2"].value_counts()

## 保存本章结果

保存表格、参数摘要和可供后续章节读取的数据。

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
keys = ["manual_coarse", "manual_level1", "manual_level2"]
ctx.table("manual_annotations", adata.obs[keys + ["samples", coarse_key, fine_key]])
for key in keys:
    ctx.table(key.replace(".", "_") + "_by_sample", pd.crosstab(adata.obs[key], adata.obs["samples"]))
ctx.finish(adata, {"fine_cell_types": adata.obs[keys[-1]].value_counts().to_dict()})


## 结果阅读与思考

请打开本次结果目录中的 summary.json、tables 和 figures。将目的、方法、结果和解释写入本章报告源稿，再更新 Word。

思考题：为什么更换聚类参数后不能照搬以前的编号映射？

运行与报告操作见课程根目录的 99_运行与AI协作指南.md。